## Setup

In [12]:
from arcgis.gis import GIS
import arcpy
from arcgis.features import FeatureLayer
from arcgis.features import GeoAccessor
import pandas as pd
import json
import urllib.request
import urllib.parse
import os
import glob
import tempfile
import requests
import time

In [3]:
gis = GIS('pro')
print(gis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
arcpy.env.addOutputsToMap = False
arcpy.env.overwriteOutput = True
project_path = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Superbloom Sandbox"
working_gdb = os.path.join(project_path, r"California Superbloom Sandbox.gdb")

GIS @ https://www.arcgis.com/ version:[2026, 2]


In [ ]:
url = "https://api.inaturalist.org/v1/observations"
base_params = {
    "swlat": 35.0, "swlng": -119.9,
    "nelat": 35.4, "nelng": -119.5,
    "iconic_taxa": "Plantae",
    "d1": "2026-02-01",
    "d2": "2026-04-30",
    "quality_grade": "research",
    "per_page": 200,
    "order_by": "id",
    "order": "asc"
}

all_observations = []
id_above = 0

while True:
    params = {**base_params, "id_above": id_above}
    response = requests.get(url, params=params)
    data = response.json()
    results = data["results"]
    
    if not results:
        break
    
    all_observations.extend(results)
    id_above = results[-1]["id"]
    print(f"Fetched {len(all_observations)} of {data['total_results']}")
    
    time.sleep(1)  # be polite to their servers, per their recommended practices

In [ ]:
records = []
for obs in all_observations:
    if obs.get("geojson") is None:
        continue  # skip anything without real coordinates
    
    lon, lat = obs["geojson"]["coordinates"]
    
    records.append({
        "id": obs["id"],
        "species": obs.get("taxon", {}).get("name", "Unknown"),
        "common_name": obs.get("taxon", {}).get("preferred_common_name", "Unknown"),
        "observed_on": obs.get("observed_on"),
        "lat": lat,
        "lon": lon
    })

df = pd.DataFrame(records)
print(len(df))

In [ ]:
print(f"{len(all_observations) - len(df)} observations had no usable coordinates")

In [ ]:
sdf = GeoAccessor.from_xy(df, x_column="lon", y_column="lat")
sdf.spatial.to_featureclass(location=os.path.join(working_gdb, "superbloom_observations"))

In [ ]:
arcpy.analysis.Clip(
    in_features=os.path.join(working_gdb, "superbloom_observations"),
    clip_features=os.path.join(working_gdb, "carrizo_Plains_Dissolved"),
    out_feature_class=os.path.join(working_gdb, "Carrizo_Observations")
)

In [ ]:
species_counts = df[df['common_name'] != 'Unknown']['common_name'].value_counts()
top_12 = species_counts.head(12).index.tolist()

print(top_12)

In [ ]:
top_12_escaped = [name.replace("'", "''") for name in top_12]
species_list = "', '".join(top_12_escaped)
where_clause = f"common_name IN ('{species_list}')"
print(where_clause)

arcpy.analysis.Select(
    in_features=os.path.join(working_gdb, "Carrizo_Observations"),
    out_feature_class=os.path.join(working_gdb, "Carrizo_top12"),
    where_clause=where_clause
)